# COMP7103 Lab 6 · Detect complaint reviews with three methods

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab6/Lab6_complaints.ipynb)

A shop and a restaurant chain want to flag complaints among their customer reviews. You have 300 reviews, each labelled complaint (1) or not (0). You try **three methods on the same 100 test reviews**: TF-IDF + logistic regression (LR; the classical baseline), sentence embeddings + LR, and a chat LLM with no examples (zero-shot), run twice. Then you compare them in one table: F1, cost and time, and how often the two LLM runs agree.

| Time | What you do |
|---|---|
| 5 min | Brief |
| 30 min | Guided work: 3 methods, 1 table |
| 10 min | Share-out |

This lab is ungraded practice for the exams. Data: UCI Sentiment Labelled Sentences (Kotzias et al., 2015, Amazon and Yelp reviews), CC BY 4.0.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup (brief, 5 min)

This cell loads the 300 reviews and their sentence embeddings (computed in advance, so nothing is downloaded from a model hub). The split is fixed: 200 training reviews and 100 test reviews, half of them complaints. Every method is scored on the same 100 test reviews.

In [ ]:
import os, time, warnings, urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# A blank (___) that you have not filled in stops with: name '___' is not defined.
# (The next two lines make sure of that. You do not need to read or change them.)
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab6/data/"

# get(name) gives the path of a data file: the copy next to the notebook if there is one, otherwise a fresh
# download from the course GitHub. You do not need to read how it works.
def get(name):
    """Use the local file if it is there (Lab 6 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 6 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab6")
    return os.path.join("data", name)

df = pd.read_csv(get("reviews_300.csv"))
emb = np.load(get("review_emb.npy")).astype(np.float32)     # row i = review i of the CSV
is_train = (df["split"] == "train").to_numpy()             # True for the 200 training reviews
is_test = (df["split"] == "test").to_numpy()               # True for the 100 test reviews
train = df[is_train]
test = df[is_test].copy()
test.index = range(1, len(test) + 1)                        # test review k is number k in the LLM message
E_train = emb[is_train]                                     # the embeddings of the training reviews
E_test = emb[is_test]
print("train:", len(train), "reviews · test:", len(test), "reviews · embeddings:", emb.shape)

## 1 · The reviews (2 min)

Short sentences from Amazon (mobile phones and accessories) and Yelp (restaurants). `complaint` is 1 for a negative review.

In [ ]:
df.head(8)

In [ ]:
# For each split (train/test) and source: the number of reviews and the share that are complaints
df.groupby(["split", "source"])["complaint"].agg(reviews="size", share_complaints="mean")

We score each method with **F1 for the complaint class**. Precision $P$ is the share of flagged reviews that are complaints; recall $R$ is the share of complaints that are flagged; $F_1 = \dfrac{2PR}{P + R}$. F1 is 1 for a perfect detector; flagging every review gives $P = 0.5$, $R = 1$, so $F_1 \approx 0.67$ here.

## 2 · Method 1: TF-IDF + logistic regression (6 min)

TF-IDF (term frequency × inverse document frequency, Lecture 2) turns each review into a vector of word weights. Logistic regression (Lecture 5) learns one weight per word from the 200 training reviews. Fill in the blanks: the vectorizer, the method that predicts, and the true labels of the test reviews. The cell also measures the time to fit and predict.

In [ ]:
t0 = time.time()
tfidf_lr = make_pipeline(___(), LogisticRegression(max_iter=1000))
tfidf_lr.fit(train["text"], train["complaint"])
pred_tfidf = tfidf_lr.___(test["text"])
secs_tfidf = time.time() - t0

f1_tfidf = f1_score(___, pred_tfidf)
print(f"TF-IDF + LR: F1 {f1_tfidf:.3f} · fit + predict {secs_tfidf:.3f} s")

## 3 · Method 2: embeddings + logistic regression (6 min)

A sentence embedding is a vector of 384 numbers from all-MiniLM-L6-v2, a small pretrained sentence-embedding model; reviews with similar meaning get nearby vectors. `E_train` and `E_test` hold them, one row per review. The same logistic regression now learns one weight per embedding number. Computing the 300 embeddings took 1.25 s on a CPU when the file was made; we add that time.

In [ ]:
EMBED_SECONDS = 1.25    # time to embed all 300 reviews on a CPU (done in advance)

t0 = time.time()
emb_lr = LogisticRegression(max_iter=1000)
emb_lr.fit(___, train["complaint"])
pred_emb = emb_lr.predict(___)
secs_emb = time.time() - t0 + EMBED_SECONDS

f1_emb = f1_score(test["complaint"], pred_emb)
print(f"Embeddings + LR: F1 {f1_emb:.3f} · fit + predict + embedding {secs_emb:.2f} s")

## 4 · Method 3: an LLM, zero-shot, twice (11 min)

Zero-shot: the LLM gets the task in words and **no labelled examples**. It sees only the 100 test reviews, never the training labels.

1. Run the next cell. It prints the message: instructions, then the 100 test reviews, numbered 1 to 100. The reviews are public; no personal data.
2. Open a **new chat** in any chat LLM you use, paste the whole message, and send it. Note roughly how many seconds the answer takes.
3. Paste the whole answer between the first pair of triple quotes in the cell after.
4. Open a **second new chat**, paste the same message again, and paste that answer between the second pair of triple quotes.

No LLM available? Use the ready-made answers (see the comments in that cell). They come from Qwen2.5-1.5B-Instruct, a small open model, given exactly this message twice, with sampling switched on (temperature 0.7) so that the two runs can differ, as two chats can.

Few-shot prompting (adding 5 labelled examples to the message) is an optional extension: if time allows, try it and see whether F1 and agreement improve.

In [ ]:
message = open(get("llm_prompt.txt")).read()
print(message)

In [ ]:
llm_reply1 = """
paste the answer from the first new chat here
"""
llm_reply2 = """
paste the answer from the second new chat here
"""
# No LLM? Use the ready-made answers instead:
# llm_reply1 = open(get("llm_fallback_run1.txt")).read()
# llm_reply2 = open(get("llm_fallback_run2.txt")).read()

LLM_SECONDS = ___   # your estimate: seconds from sending the message to the end of the answer (one run)
# No LLM? The ready-made answers took 330 s and 940 s on a shared 2-core CPU: LLM_SECONDS = 330

The helpers below (given) read the labels from an answer: `read_label()` reads one line such as `7: 1`, and `parse_labels()` uses it on every line. They accept small differences in format, and tell you if a review has no label.

In [ ]:
def read_label(line):
    """Read "<review number> <label>" from one piece of text. Returns (number, label), or None if there is no label."""
    line = line.lower()
    for symbol in ["*", "|", "#", ">", ":", ".", ",", ")", "(", "[", "]", "=", "-", "–", "—", "→", "•", "·"]:
        line = line.replace(symbol, " ")         # punctuation becomes a space
    words = line.split()                         # e.g. ["review", "7", "label", "complaint"]
    words = [w for w in words if w not in ("review", "label", "answer")]
    if len(words) < 2 or not words[0].isdigit():
        return None                              # it does not start with a review number
    number = int(words[0])
    first = words[1]                             # the word right after the number
    if first == "1" or first.startswith("complaint") or first == "yes":
        return number, 1
    if first == "0" or first in ("not", "no", "non"):
        return number, 0
    if words[-1] in ("0", "1"):                  # the review text was repeated: the label is at the end
        return number, int(words[-1])
    return None

def parse_labels(reply, n):
    """Read one label per review from an LLM answer; return an array of n labels (1 = complaint).

    Accepts lines such as "7: 1", "**7:** 0", "7. Complaint", "7 – Not complaint", "Review 7: Label: complaint",
    "| 7 | 1 |", or labels run together on one line ("1: 0, 2: 1, ...").
    A review with no label is counted as 0 (not a complaint) and listed, so you can see it."""
    found = {}                                   # review number -> label
    pieces = reply.splitlines()                  # usually one review per line
    if len(pieces) < n:                          # labels run together on one line, e.g. "1: 0, 2: 1, ..."
        pieces = reply.replace("\n", ",").split(",")
    for piece in pieces:
        result = read_label(piece)
        if result is None:
            continue
        number, label = result
        if 1 <= number <= n and number not in found:
            found[number] = label

    labels = []                                  # one label per review, in order
    missing = []
    for k in range(1, n + 1):
        if k in found:
            labels.append(found[k])
        else:
            labels.append(0)                     # no label: counted as 0
            missing.append(k)
    print(f"Read {len(found)} of {n} labels.")
    if missing:
        print(f"No label for reviews {missing}: counted as 0.")
    if len(found) < 0.9 * n:
        print("WARNING: most labels were not read. Check that you pasted the whole answer, one line per review.")
    return np.array(labels)

Score both runs against the true labels, then measure **agreement**: the percentage of the 100 test reviews on which run 1 and run 2 give the same label.

In [ ]:
labels1 = parse_labels(llm_reply1, n=len(test))
labels2 = parse_labels(llm_reply2, n=len(test))

f1_llm1 = f1_score(test["complaint"], labels1)
f1_llm2 = f1_score(test["complaint"], labels2)
agreement = 100 * np.mean(___ == ___)
print(f"LLM zero-shot: F1 run 1 {f1_llm1:.3f} · F1 run 2 {f1_llm2:.3f} · runs agree on {agreement:.0f}% of the reviews")

## 5 · One table: F1, cost and time (5 min)

**Cost.** An LLM provider charges per token (a word or part of a word), with one price for the input (the message) and a higher one for the output (the answer). A rule of thumb for English: 1 token ≈ 4 characters. The prices below are the lecture's **example values, not real quotes: replace them with your provider's current prices**. Methods 1 and 2 run on your own computer, so their running cost is ≈ 0.

In [ ]:
PRICE_IN = 3.0     # EXAMPLE only: US$ per 1 million input tokens. Replace with your provider's price.
PRICE_OUT = 15.0   # EXAMPLE only: US$ per 1 million output tokens. Replace with your provider's price.

def n_tokens(text):
    """Rough token count: about 4 characters per token."""
    return len(text) / ___

cost_llm = (n_tokens(message) * PRICE_IN + n_tokens(llm_reply1) * PRICE_OUT) / ___   # one run, 100 reviews
print(f"Message ≈ {n_tokens(message):,.0f} tokens · answer ≈ {n_tokens(llm_reply1):,.0f} tokens · one run ≈ US$ {cost_llm:.4f}")

Now the comparison table, for the 100 test reviews (the LLM row uses run 1). Fill in the F1 column.

In [ ]:
table = pd.DataFrame({
    "F1": [f1_tfidf, ___, ___],
    "Cost, US$": [0.0, 0.0, cost_llm],
    "Time, s": [secs_tfidf, secs_emb, LLM_SECONDS],
    "Labels used": [len(train), len(train), 0],
}, index=["TF-IDF + LR", "Embeddings + LR", "LLM zero-shot"])
print(f"LLM run 1 vs run 2 agreement: {agreement:.0f}%")
table.round({"F1": 3, "Cost, US$": 4, "Time, s": 2})

The costs look tiny for 100 reviews. Scale them up to 100,000 reviews, the lecture's example. Two things to keep in mind. Here one message carries 100 reviews, so the instructions are paid once per 100 reviews; the lecture's calculator assumes one call per review, which costs more. And methods 1 and 2 needed 200 labelled reviews; at the lecture's example price of US$ 0.10 per human label, that is a one-off cost.

In [ ]:
N = 100_000
LABEL_PRICE = 0.10   # EXAMPLE only: US$ per human label

# The LLM: one run covers the 100 test reviews, so scale cost and time by N / 100
llm_cost_N = cost_llm * N / len(test)
llm_hours_N = LLM_SECONDS * N / len(test) / 3600
print(f"LLM zero-shot for {N:,} reviews ≈ US$ {llm_cost_N:,.0f} and ≈ {llm_hours_N:,.1f} h at your speed (one chat)")

# Embeddings + LR: pay once for the 200 training labels; embedding 300 reviews took EMBED_SECONDS
label_cost = LABEL_PRICE * len(train)
embed_minutes_N = EMBED_SECONDS * N / len(df) / 60
print(f"Embeddings + LR: one-off US$ {label_cost:.0f} for {len(train)} labels, then ≈ 0; "
      f"embedding {N:,} reviews ≈ {embed_minutes_N:,.0f} min on a CPU")

# After how many reviews does the LLM cost more than the labels?
llm_cost_per_review = cost_llm / len(test)
break_even = round(label_cost / llm_cost_per_review, -3)          # rounded to the nearest 1,000
print(f"The labels pay for themselves after ≈ {break_even:,.0f} reviews")

Where do the methods differ? The test reviews on which the two LLM runs disagree, with every method's label (1 = complaint):

In [ ]:
view = test[["text", "complaint"]].copy()     # the review and its true label
view["tfidf"] = pred_tfidf                     # each method's label, one column per method
view["emb"] = pred_emb
view["llm1"] = labels1
view["llm2"] = labels2
view[view["llm1"] != view["llm2"]]             # keep the reviews where the two LLM runs disagree

**Your choice.** The shop must flag complaints in 100,000 new reviews every month. Which method would you use, and why? One sentence, with numbers from your table: F1, cost, time, and agreement.

In [ ]:
CHOICE = "TF-IDF + LR / Embeddings + LR / LLM zero-shot"
WHY = "one sentence, with numbers from the table"
print(CHOICE, "·", WHY)

## 6 · Share-out (10 min)

- Report your table: **F1, cost (US$) and time (s) for each of the three methods**, and your **LLM run 1 vs run 2 agreement (%)**. We collect them on the class board, which names the best method and its gain in F1 over the TF-IDF baseline.
- Which methods beat 0.67, the F1 of flagging every review (section 1)? Why can TF-IDF + LR with 200 short reviews fall below it?
- Which chat LLM did you use? Did it follow the format, label all 100 reviews, and give the same labels in both chats?
- Which method would you choose for 100,000 reviews a month, and would your choice change if the reviews could not leave the company?
- The habit from the lecture: every route is measured against the classical baseline, on the same test set.